# 01b - Experimenting with Attribute Combinations

This is step 4 of the class workflow, same as the "Experimenting with Attribute Combinations" section of the California housing example. There, the class built `rooms_per_house` and `bedrooms_ratio` out of raw columns and checked whether the new combos correlate with house value better than the raw columns did.

Here we do the same thing for points. After the EDA (`01_eda.ipynb`) we added combined features to `src/build_features.py` (true shooting, blends with last season, matchup history relative to the player's average, and more). This notebook checks them with the same tool the class used: **sorted correlation with the target, on the training set only.**

No modeling yet. The test season (2025-26) is never touched.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from build_features import FEATURES, NEW_FEATURES  # same lists the models will use

pd.set_option("display.max_rows", 60)

df = pd.read_csv(ROOT / "data" / "processed" / "features.csv", parse_dates=["GAME_DATE"])

# training seasons only (same split as the EDA): test = 2025-26, 2026-27 = live site only
train_set = df[df["SEASON"].isin(["2023-24", "2024-25"])].copy()
print(f"train rows: {len(train_set):,}  |  features: {len(FEATURES)}  |  new this pass: {len(NEW_FEATURES)}")

train rows: 38,468  |  features: 33  |  new this pass: 10


### The new features

| feature | what it is |
|---|---|
| `tsa_season`, `tsa_l5` | shooting volume: true shooting attempts per game (FGA + 0.44 x FTA), season and last 5. Replaces the separate FGA / 3PA / FTA averages |
| `ts_season` | shooting efficiency: season points / (2 x season TSA), from totals |
| `ts_hot` | shooting hot or cold lately: last-5 TS% minus season TS%, shrunk toward 0 when there are few games |
| `pts_blend`, `min_blend` | season average shrunk toward last season's average early on (last season counts like 10 games) |
| `pts_vs_opp_rel` | how much above or below his own average he scored in past games vs this opponent (this season + last), shrunk toward 0 |
| `games_missed` | length of his most recent absence (until he's 10 games back) |
| `starter_proxy` | 1 if he's averaging 28+ minutes over his last 5 games |
| `abs_net_rtg_gap` | how lopsided the matchup is, either direction (blowout risk) |

The opponent ratings (`opp_def_rtg`, `opp_pace`, `opp_pts_allowed`, `net_rtg_gap`) kept their names but are now blended with last season too, so they aren't counted as new here.

## 1. Correlation with points

Same as `corr_matrix["median_house_value"].sort_values(ascending=False)` in the class example. A `*` marks the new features.

In [2]:
corr_matrix = train_set[FEATURES + ["PTS"]].corr()
corr_pts = corr_matrix["PTS"].drop("PTS").sort_values(ascending=False)

def mark_new(s):
    return pd.DataFrame({"corr": s.round(3), "new": ["*" if f in NEW_FEATURES else "" for f in s.index]})

mark_new(corr_pts)

,corr,new
pts_blend,0.678,*
tsa_season,0.674,*
pts_season,0.674,
pts_l10,0.672,
tsa_l5,0.669,*
pts_l5,0.654,
pts_l3,0.630,
pts_prev_season,0.627,
min_blend,0.563,*
min_season,0.561,


**Takeaway:** The top of the list is "how much does he usually score", and the new `pts_blend` (0.678) edges out every old version of it, including the plain season average (0.674). Shot volume (`tsa_season`, `tsa_l5`, ~0.67) is just as strong as past points, which makes sense: you can't score without shooting. Minutes come next (~0.55), and `starter_proxy` (0.47) captures most of that with a single yes/no.

Everything else is tiny on its own (under 0.1). That doesn't mean useless. This view is dominated by the gap between stars and bench guys, so anything that isn't "how good is he" looks weak here. The next check fixes that.

## 2. Correlation with points *above his own average*

A feature that just says "this guy is a star" will correlate with PTS but adds nothing, because `pts_season` already says that. So we subtract his season average and ask: **which features explain why he scored more or less than usual tonight?** That's the part a model has to get right to beat the baseline.

In [3]:
train_set["PTS_vs_avg"] = train_set["PTS"] - train_set["pts_season"]

corr_vs_avg = (train_set[FEATURES + ["PTS_vs_avg"]].corr()["PTS_vs_avg"]
               .drop("PTS_vs_avg").sort_values(ascending=False))
mark_new(corr_vs_avg)

,corr,new
team_missing_min,0.145,
min_trend,0.083,
opp_pts_allowed,0.055,
opp_def_rtg,0.051,
min_l3,0.039,
opp_pace,0.037,
games_last_7,0.037,
opp_missing_min,0.036,
pts_vs_opp_rel,0.035,*
games_played,0.032,


**Takeaway:** The ranking flips. The top feature is now `team_missing_min` (0.145): when teammates are out, he scores above his average, which is exactly the "usage goes up when teammates sit" idea the pros use. Next is `min_trend` (0.083, role growing), then the opponent stats (`opp_pts_allowed`, `opp_def_rtg`, ~0.05: bad defenses give up more).

The new features are mostly small but pointing the right way:
- `pts_vs_opp_rel` (+0.035) and `ts_hot` (+0.027): small, positive signals. Matchup history and hot shooting carry a little real information once they're shrunk for small samples.
- `ts_season` (-0.103) is the most negative feature. Very efficient shooters tend to score *below* their average next game. That's regression to the mean: some of a high TS% is luck that doesn't last.
- `pts_season` itself is negative (-0.072) for the same reason, and `pts_blend` is less negative (-0.044). The blend already corrects some of that overreaction.
- `abs_net_rtg_gap` is basically 0. See section 4.

## 3. Do the blends help where they should? (early season)

The blends exist to fix the start of the season, when "season average" is based on 1 to 5 games. So compare them where it matters: a player's first 5 and first 10 games, and the opponent's first 10 games.

For the opponent we rebuild the old unblended rating (season-to-date only) from the team logs, just for this comparison.

In [4]:
rows = []
for label, sub in [("all games", train_set),
                   ("his first 10 games", train_set[train_set["games_played"] < 10]),
                   ("his first 5 games", train_set[train_set["games_played"] < 5])]:
    rows.append({"subset": label, "rows": len(sub),
                 "pts_season": sub["pts_season"].corr(sub["PTS"]),
                 "pts_blend": sub["pts_blend"].corr(sub["PTS"]),
                 "pts_prev_season": sub["pts_prev_season"].corr(sub["PTS"])})
print("correlation with PTS")
pd.DataFrame(rows).set_index("subset").round(3)

correlation with PTS


,rows,pts_season,pts_blend,pts_prev_season
subset,,,,
all games,38468,0.674,0.678,0.627
his first 10 games,5963,0.664,0.696,0.671
his first 5 games,2936,0.637,0.690,0.673


In [5]:
from build_features import build_team_features, load_logs

# old version: opponent defensive rating from this season's games only (no blend)
team_feats = build_team_features(load_logs("team"))
raw = team_feats[["GAME_ID", "TEAM_ABBREVIATION", "def_rtg", "team_games_before"]].rename(
    columns={"TEAM_ABBREVIATION": "OPP", "def_rtg": "opp_def_rtg_raw",
             "team_games_before": "opp_games_before"})
early = train_set.merge(raw, on=["GAME_ID", "OPP"])
early = early[early["opp_games_before"].between(1, 9)]   # raw is blank on game 1

print(f"opponent's games 2-10 ({len(early):,} rows)")
print("  range, raw:     ", early["opp_def_rtg_raw"].min().round(1), "to", early["opp_def_rtg_raw"].max().round(1))
print("  range, blended: ", early["opp_def_rtg"].min().round(1), "to", early["opp_def_rtg"].max().round(1))
pd.DataFrame({
    "corr with PTS_vs_avg": [early["opp_def_rtg_raw"].corr(early["PTS_vs_avg"]),
                             early["opp_def_rtg"].corr(early["PTS_vs_avg"])]},
    index=["opp_def_rtg raw (old)", "opp_def_rtg blended (new)"]).round(3)

opponent's games 2-10 (4,244 rows)
  range, raw:      82.4 to 143.8
  range, blended:  101.9 to 117.6


,corr with PTS_vs_avg
opp_def_rtg raw (old),0.034
opp_def_rtg blended (new),0.054


**Takeaway:** The blends do their job early in the season.
- Player side: in his first 5 games, `pts_season` drops to 0.637 (it's based on a handful of games), while `pts_blend` holds at 0.690, better than either this season or last season alone. Over the full season the two are nearly the same, which is the point: the blend fades out as games pile up.
- Opponent side: in a team's games 2 to 10, the raw defensive rating ranges from 82 to 144 (one blowout makes a team look historically good or bad). Blended, it stays between 102 and 118, a believable range, and its correlation with points-above-average goes up from 0.034 to 0.054.

## 4. Bonus: what moves *minutes*? (preview of stage 1 of the two-stage model)

`abs_net_rtg_gap` and `starter_proxy` were added for minutes reasons (blowouts sit starters), so check them against minutes above his average.

In [6]:
train_set["MIN_vs_avg"] = train_set["MIN"] - train_set["min_season"]

min_cols = ["team_missing_min", "min_trend", "games_since_return", "starter_proxy",
            "abs_net_rtg_gap", "net_rtg_gap", "games_missed"]
corr_min = train_set[min_cols + ["MIN_vs_avg"]].corr()["MIN_vs_avg"].drop("MIN_vs_avg")
print(mark_new(corr_min.sort_values(ascending=False)))

# average minutes above his norm, by how lopsided the matchup looks
print(train_set.groupby(pd.cut(train_set["abs_net_rtg_gap"], [0, 3, 6, 10, 30]), observed=True)
      ["MIN_vs_avg"].mean().round(2))

                     corr new
team_missing_min    0.238    
min_trend           0.207    
games_since_return  0.062    
starter_proxy       0.045   *
abs_net_rtg_gap    -0.005   *
net_rtg_gap        -0.044    
games_missed       -0.045   *
abs_net_rtg_gap
(0, 3]      0.57
(3, 6]      0.50
(6, 10]     0.57
(10, 30]    0.42
Name: MIN_vs_avg, dtype: float64


**Takeaway:** Minutes above his average are driven by teammates being out (`team_missing_min`, 0.24) and a growing role (`min_trend`, 0.21). Those are much stronger signals for minutes than for points, which supports the two-stage idea: predict minutes first, where these features are strong.

The blowout feature is weak: in the most lopsided matchups (10+ net rating gap) players average only ~0.15 fewer minutes vs their norm than in close ones. Pregame ratings just don't predict blowouts well. We keep it and let cross-validation decide.

## Key findings

- **Points mostly come from the obvious stuff:** his scoring average, shot volume, and minutes (correlations 0.55 to 0.68). Every model has to start there.
- **The interesting part is beating his own average.** Once you subtract his average, the biggest signal is teammates being out (0.15 for points, 0.24 for minutes). That's the usage effect real projection systems rely on, and it's why the site has a "who's out" checklist.
- **Blending in last season fixes the early season.** In a player's first 5 games, the blended average correlates 0.69 with points vs 0.64 for this season alone, and opponent ratings stop swinging wildly.
- **Hot shooting and matchup history are weak signals** (~0.03 each, after shrinking for small samples). Meanwhile a high season-long TS% predicts scoring slightly *below* average next game (-0.10): some of that efficiency is luck, and it regresses to the mean.
- **Correlation only sees straight-line effects.** Small numbers here don't kill a feature. Gradient boosting can find combinations, and cross-validation gets the final say in the models notebook.